# Title

## Table of Contents <a id='back'></a>
- [Project Introduction](#project-introduction)
    - [Analysis Objectives](#analysis-objectives)
    - [Executive Summary](#executive-summary)
- [Importing Libraries and Opening Data Files](#importing-libraries-and-opening-data-files)
- [Data Wrangling](#data-wrangling)
    - [Duplicates](#duplicates)
    - [Missing Values](#missing-values)
    - [Data Transformation](#data-transformation)
- [Exploratory Data Analysis](#exploratory-data-analysis)
- [Conclusions and Reccomendations](#conclusions-and-reccomendations)
- [Dataset Citation](#dataset-citation)

## Project Introduction

[project intro]

### Analysis Objectives

[Analysis Objectives]

### Executive Summary

[Results]


[Back to Table of Contents](#back)

## Importing Libraries and Opening Data Files

In [1]:
# Importing the needed libraries for this assignment
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

In [2]:
# Importing file for assignment
try:
    df = pd.read_csv('xxxecom_orders_data.csv', sep=',')
except:
    df = pd.read_csv('/datasets/xxxecom_orders_data.csv', sep=',')

try:
    df = pd.read_csv('xxxecom_orders_data.csv', sep=',')
except:
    df = pd.read_csv('/datasets/xxxecom_orders_data.csv', sep=',')

[Back to Table of Contents](#back)

## Data Wrangling

In [3]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 987 entries, 0 to 986
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          966 non-null    float64
 1   customer_id       965 non-null    float64
 2   age               987 non-null    int64  
 3   city              987 non-null    object 
 4   state             987 non-null    object 
 5   product_category  987 non-null    object 
 6   sub_category      987 non-null    object 
 7   color             987 non-null    object 
 8   price             987 non-null    float64
 9   Quantity          987 non-null    int64  
 10  order_date        968 non-null    object 
 11  ship_date         968 non-null    object 
dtypes: float64(3), int64(2), object(7)
memory usage: 92.7+ KB


,order_id,customer_id,age,city,state,product_category,sub_category,color,price,Quantity,order_date,ship_date
0,583.0,145.0,48,Los Angeles,California,outerwear,hoodie,Blue,45.99,1,1/20/2025,1/26/2025
1,269.0,197.0,23,Young America,Minnesota,clothing,shorts,Black,23.99,5,4/14/2025,4/18/2025
2,962.0,906.0,36,Minneapolis,Minnesota,outerwear,hoodie,White,45.99,3,12/9/2025,12/16/2025
3,106.0,754.0,32,Fresno,California,accessories,hat,Black,15.99,1,1/16/2025,1/23/2025
4,902.0,31.0,39,Moreno Valley,California,clothing,socks,Black,11.99,2,12/7/2025,12/11/2025


Observation:

- The order_id and customer_id column are float data types and could be converted int data type to lower data usage

- Across all the string data type columns are in mixed lowercase and proper case format and should all be converted to snake case format for consistency and remove possible leading or trailing white spaces

- The order_date and ship_date columns are in string and should be converted to datetime format to lower data usage

### Duplicates

In [4]:
# Checking for duplicates
df.duplicated().sum()

0

Observation:

- There are no repeat rows which is good 

- Since the order_id column is the unique identifier I will check that it also has no duplicates

In [5]:
# Counting number of possible duplicated rows
print(f'{df['order_id'].duplicated().sum()} duplicate order_id')

# Counting the unique identifier column for unqiue values
print(f'{df['order_id'].nunique()} unique values')


367 duplicate order_id
619 unique values


Observation:

- However, when looking into the unique identifier column there are only 619 unique values and 367 duplicate order_id values

- One possibility is that some orders could repeat if there are multiple different products in the same order but this could still be a problem and needs further inspection

In [6]:
# Looking more into the order_id column
df[df['order_id'].duplicated(keep=False)].sort_values(by='order_id').head(30)

,order_id,customer_id,age,city,state,product_category,sub_category,color,price,Quantity,order_date,ship_date
431,8.0,NaN,44,Berkeley,California,accessories,hat,Gray,15.99,1,10/29/2025,11/4/2025
280,8.0,659.0,34,Pocatello,Idaho,clothing,socks,Black,11.99,1,11/7/2025,11/14/2025
924,12.0,90.0,34,Detroit,Michigan,clothing,sweater,Blue,33.99,3,7/11/2025,7/14/2025
397,12.0,786.0,32,Sacramento,California,clothing,t-shirt,White,17.99,4,8/11/2025,8/18/2025
240,15.0,884.0,53,Colorado Springs,Colorado,clothing,t-shirt,White,17.99,2,9/27/2025,9/29/2025
422,15.0,774.0,34,San Jose,California,clothing,socks,White,11.99,2,3/15/2025,3/21/2025
769,15.0,513.0,39,Madison,Wisconsin,clothing,pants,Blue,37.99,2,12/24/2025,12/28/2025
650,17.0,NaN,65,Torrance,California,clothing,shirt,Gray,25.99,1,1/2/2025,1/9/2025
753,17.0,259.0,33,Long Beach,California,clothing,pants,Navy,37.99,2,1/1/2025,1/8/2025
7,17.0,579.0,36,Boulder,Colorado,outerwear,hoodie,Black,45.99,2,3/13/2025,3/18/2025


Observation:

- Looking at the first 30 rows many order_id values repeat as many as 3 times

- Despite having the same order_id the age, address locations, and order dates are different indicating that these should be different orders

- Additionally, there is a null value in the customer_id column indicating that later I need to fix null values

In [7]:
# Replacing all order_id values 
df['order_id'] = range(1, len(df) + 1)

# Counting number of possible duplicated rows
print(f'{df['order_id'].duplicated().sum()} duplicate order_id')
print(df['order_id'].head())

0 duplicate order_id
0    1
1    2
2    3
3    4
4    5
Name: order_id, dtype: int64


Observation:

- Removed all order_id duplicate values

In [8]:
# Looking into duplicate customer_id values
df['customer_id'].duplicated().sum()

370

Observation:

- There are 370 duplicate customer_id values which may need to be replaced or fixed

- These could be repeat customers so I need to be careful of which values to remove

In [9]:
# Looking at customer_id column for unique customers
df[df['customer_id'].duplicated(keep=False)].sort_values(by='customer_id').head(30)

,order_id,customer_id,age,city,state,product_category,sub_category,color,price,Quantity,order_date,ship_date
423,424,7.0,29,Minneapolis,Minnesota,clothing,sweater,Blue,33.99,1,6/2/2025,6/5/2025
22,23,7.0,29,Minneapolis,Minnesota,accessories,hat,Beige,15.99,3,7/25/2025,8/1/2025
293,294,14.0,48,Littleton,Colorado,clothing,shirt,Black,25.99,4,3/31/2025,4/2/2025
972,973,14.0,48,Littleton,Colorado,accessories,hat,Navy,15.99,3,2/25/2025,2/27/2025
610,611,15.0,59,Denver,Colorado,clothing,shirt,Gray,25.99,1,8/1/2025,8/7/2025
433,434,15.0,59,Denver,Colorado,outerwear,hoodie,Blue,45.99,1,1/25/2025,1/26/2025
9,10,17.0,31,Saint Paul,Minnesota,clothing,shirt,Black,25.99,1,2/10/2025,2/12/2025
459,460,17.0,31,Saint Paul,Minnesota,outerwear,hoodie,Black,45.99,3,10/31/2025,11/4/2025
502,503,18.0,26,Bakersfield,California,outerwear,hoodie,Black,45.99,3,10/2/2025,10/5/2025
50,51,18.0,26,Bakersfield,California,clothing,t-shirt,Navy,17.99,1,7/12/2025,7/17/2025


Observation:

- Looking more into the customer_id duplicates it shows each row has the same age, city, and state which means its a different order from the same customer

- No fixes are needed for this column and the duplicates can remain the same

[Back to Table of Contents](#back)

### Missing Values

In [10]:
# Checking for null values
df.isna().sum()

order_id             0
customer_id         22
age                  0
city                 0
state                0
product_category     0
sub_category         0
color                0
price                0
Quantity             0
order_date          19
ship_date           19
dtype: int64

Observation:

- Looking into the null values I need to look into the customer_id, order_date, and shipping date columns

In [11]:
#
df[df['customer_id'].isna()]

,order_id,customer_id,age,city,state,product_category,sub_category,color,price,Quantity,order_date,ship_date
38,39,NaN,37,Sacramento,California,clothing,socks,Black,11.99,1,3/10/2025,3/16/2025
65,66,NaN,22,Seattle,Washington,clothing,socks,White,11.99,3,11/5/2025,11/6/2025
121,122,NaN,57,Monticello,Minnesota,clothing,socks,White,11.99,4,5/7/2025,5/13/2025
188,189,NaN,51,Sacramento,California,clothing,socks,White,11.99,2,5/19/2025,5/20/2025
267,268,NaN,28,Van Nuys,California,clothing,socks,White,11.99,2,10/31/2025,11/6/2025
285,286,NaN,34,Minneapolis,Minnesota,clothing,shirt,Navy,25.99,3,NaN,NaN
310,311,NaN,38,Fresno,California,accessories,hat,Blue,15.99,1,7/23/2025,7/27/2025
341,342,NaN,28,Milwaukee,Wisconsin,clothing,pants,Navy,37.99,2,7/24/2025,7/30/2025
405,406,NaN,62,Pomona,California,clothing,t-shirt,White,17.99,2,4/26/2025,4/30/2025
425,426,NaN,28,Long Beach,California,clothing,t-shirt,Black,17.99,2,3/31/2025,4/7/2025


Observation:

- Since there are only 22 null values I can remove since its only 2.2% of the total rows but I want to save as many of them as I can

- I will assign new customer id values to these null columns to retain as much sales data as I can since these are all real orders

In [12]:
#
start = int(df['customer_id'].max()) + 1
#
df.loc[df['customer_id'].isna(), 'customer_id'] = range(start, start + df['customer_id'].isna().sum())

#
print(f'{df['customer_id'].isna().sum()} null values')
print(f'{df['customer_id'].max()} max customer_id')

0 null values
1008.0 max customer_id


Observation:

- Retained all sales orders as new customer_id values

In [13]:
#
df[df['order_date'].isna()]

,order_id,customer_id,age,city,state,product_category,sub_category,color,price,Quantity,order_date,ship_date
14,15,930.0,37,Petaluma,California,outerwear,jacket,Black,59.99,1,NaN,NaN
72,73,533.0,39,Spokane,Washington,clothing,t-shirt,Black,17.99,2,NaN,NaN
79,80,840.0,43,Seattle,Washington,outerwear,hoodie,Navy,45.99,2,NaN,NaN
91,92,871.0,43,Sacramento,California,clothing,t-shirt,White,17.99,1,NaN,NaN
200,201,440.0,23,Colorado Springs,Colorado,clothing,sweater,Gray,33.99,1,NaN,NaN
230,231,422.0,28,Boise,Idaho,accessories,backpack,Navy,39.99,1,NaN,NaN
285,286,992.0,34,Minneapolis,Minnesota,clothing,shirt,Navy,25.99,3,NaN,NaN
290,291,727.0,40,Fresno,California,outerwear,hoodie,Black,45.99,1,NaN,NaN
299,300,965.0,54,Los Angeles,California,clothing,sweater,Black,33.99,1,NaN,NaN
354,355,517.0,26,Vancouver,Washington,accessories,hat,Beige,15.99,3,NaN,NaN


Observation:

- It appears that all the order date and shipping date values are missing together

- For this missing data it would be better to remove it since these could be considered missing or canceled orders

In [14]:
# Since the order and shipping dates are correlated I just need to remove all the order_date null values
df = df.dropna(subset=['order_date'])

# Checking for null values
df.isna().sum()

order_id            0
customer_id         0
age                 0
city                0
state               0
product_category    0
sub_category        0
color               0
price               0
Quantity            0
order_date          0
ship_date           0
dtype: int64

Observation:

- All null values are removed

[Back to Table of Contents](#back)

### Data Transformation

In [15]:
# Getting general information about the dataset
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 968 entries, 0 to 986
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          968 non-null    int64  
 1   customer_id       968 non-null    float64
 2   age               968 non-null    int64  
 3   city              968 non-null    object 
 4   state             968 non-null    object 
 5   product_category  968 non-null    object 
 6   sub_category      968 non-null    object 
 7   color             968 non-null    object 
 8   price             968 non-null    float64
 9   Quantity          968 non-null    int64  
 10  order_date        968 non-null    object 
 11  ship_date         968 non-null    object 
dtypes: float64(2), int64(3), object(7)
memory usage: 98.3+ KB


,order_id,customer_id,age,city,state,product_category,sub_category,color,price,Quantity,order_date,ship_date
0,1,145.0,48,Los Angeles,California,outerwear,hoodie,Blue,45.99,1,1/20/2025,1/26/2025
1,2,197.0,23,Young America,Minnesota,clothing,shorts,Black,23.99,5,4/14/2025,4/18/2025
2,3,906.0,36,Minneapolis,Minnesota,outerwear,hoodie,White,45.99,3,12/9/2025,12/16/2025
3,4,754.0,32,Fresno,California,accessories,hat,Black,15.99,1,1/16/2025,1/23/2025
4,5,31.0,39,Moreno Valley,California,clothing,socks,Black,11.99,2,12/7/2025,12/11/2025


Observation:

- Based on the meta data there are several improvements that can be made:

    1) The column names and the table elements can be converted to snakecase format for consistency and readability

    2) The customer_id column can be converted to an integer data type to reduce data usage

    3) The date columns can be converted to datetime format

In [16]:
# Checking for snakecase format
df.columns

Index(['order_id', 'customer_id', 'age', 'city', 'state', 'product_category',
       'sub_category', 'color', 'price', 'Quantity', 'order_date',
       'ship_date'],
      dtype='object')

In [17]:
# Renaming column names to snake_case format
df = df.rename(columns={'Quantity': 'quantity'})
df.columns

Index(['order_id', 'customer_id', 'age', 'city', 'state', 'product_category',
       'sub_category', 'color', 'price', 'quantity', 'order_date',
       'ship_date'],
      dtype='object')

In [18]:
# Converting string data value to datetime data type
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

# Converting the customer_id column from float to int to reduce data usage
df['customer_id'] = df['customer_id'].astype('int')

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 968 entries, 0 to 986
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          968 non-null    int64         
 1   customer_id       968 non-null    int32         
 2   age               968 non-null    int64         
 3   city              968 non-null    object        
 4   state             968 non-null    object        
 5   product_category  968 non-null    object        
 6   sub_category      968 non-null    object        
 7   color             968 non-null    object        
 8   price             968 non-null    float64       
 9   quantity          968 non-null    int64         
 10  order_date        968 non-null    datetime64[ns]
 11  ship_date         968 non-null    datetime64[ns]
dtypes: datetime64[ns](2), float64(1), int32(1), int64(3), object(5)
memory usage: 94.5+ KB


In [19]:
# Converting all elements into snakecase format and removing all nonlegible characters
for column in df.columns:
    if df[column].dtype == 'object':
        df[column] = df[column].str.lower()
        df[column] = df[column].str.replace(' ', '_')

df.head()

,order_id,customer_id,age,city,state,product_category,sub_category,color,price,quantity,order_date,ship_date
0,1,145,48,los_angeles,california,outerwear,hoodie,blue,45.99,1,2025-01-20,2025-01-26
1,2,197,23,young_america,minnesota,clothing,shorts,black,23.99,5,2025-04-14,2025-04-18
2,3,906,36,minneapolis,minnesota,outerwear,hoodie,white,45.99,3,2025-12-09,2025-12-16
3,4,754,32,fresno,california,accessories,hat,black,15.99,1,2025-01-16,2025-01-23
4,5,31,39,moreno_valley,california,clothing,socks,black,11.99,2,2025-12-07,2025-12-11


In [ ]:
# Looking for leading and trailing underscores and misspelled names
df['city'].unique()

array(['los_angeles', 'young_america', 'minneapolis', 'fresno',
       'moreno_valley', 'denver', 'richmond', 'boulder', 'grand_rapids',
       'saint_paul', 'north_hollywood', 'san_jose', 'portland',
       'sacramento', 'lansing', 'vancouver', 'santa_monica', 'ventura',
       'riverside', 'redwood_city', 'san_francisco', 'boise', 'detroit',
       'mountain_view', 'seattle', 'spokane', 'san_diego', 'eugene',
       'inglewood', 'madison', 'tacoma', 'monticello', 'bakersfield',
       'colorado_springs', 'milwaukee', 'stockton', 'rochester',
       'newport_beach', 'appleton', 'flint', 'fullerton', 'muskegon',
       'irvine', 'pasadena', 'salem', 'orange', 'alhambra', 'oceanside',
       'san_bernardino', 'anaheim', 'berkeley', 'green_bay',
       'chula_vista', 'northridge', 'idaho_falls', 'torrance',
       'san_rafael', 'oakland', 'littleton', 'whittier', 'glendale',
       'garden_grove', 'duluth', 'chico', 'pocatello', 'van_nuys',
       'palmdale', 'visalia', 'long_beach', 'sa

In [ ]:
# Looking for leading and trailing underscores and misspelled names
df['state'].unique()

array(['california', 'minnesota', 'colorado', 'michigan', 'oregon',
       'washington', 'idaho', 'wisconsin'], dtype=object)

In [ ]:
# Looking for leading and trailing underscores and misspelled names
df['product_category'].unique()

array(['outerwear', 'clothing', 'accessories'], dtype=object)

In [ ]:
# Looking for leading and trailing underscores and misspelled names
df['sub_category'].unique()

array(['hoodie', 'shorts', 'hat', 'socks', 't-shirt', 'shirt', 'jacket',
       'sweater', 'backpack', 'pants'], dtype=object)

In [ ]:
# Looking for leading and trailing underscores and misspelled names
df['color'].unique()

array(['blue', 'black', 'white', 'navy', 'gray', 'beige'], dtype=object)

Observation:

- All string columns values seem to have to apparent spelling errors

[Back to Table of Contents](#back)

In [ ]:
# Saving the new cleaned dataset if needed
df.to_csv('cleaned_data.csv', index=False) 

## Exploratory Data Analysis

Observation:

- 

[Back to Table of Contents](#back)

## Conclusions and Reccomendations

[Back to Table of Contents](#back)

## Dataset Citation

syntax:
[Dataset creator's name]. ([Year &amp; Month of dataset creation]). [Name of the dataset], [Version of the dataset]. Retrieved [Date Retrieved] from [Kaggle](URL of the dataset).

example:
Tatman, R. (2017, November). R vs. Python: The Kitchen Gadget Test, Version 1. Retrieved December 20, 2017 from https://www.kaggle.com/rtatman/r-vs-python-the-kitchen-gadget-test.

[Back to Table of Contents](#back)